## **1. Environment Setup**

In [1]:
#Setting up the environment for Colab or local execution
import sys
import os

if 'google.colab' in sys.modules:
    print("Running in Colab. Setting up repository...")
    repo_name = 'Multilingual-Classification-of-Urgent-Disaster-Response-Messages'

    #Cloning if it doesn't exist yet
    if not os.path.exists(repo_name):
        !git clone https://github.com/Rizzwan285/Multilingual-Classification-of-Urgent-Disaster-Response-Messages.git

    #Change directory into the repo
    if os.path.exists(repo_name):
        %cd {repo_name}
else:
    print("Running locally. Skipping clone.")

#Installing the required packages
!pip install -q -r requirements.txt

Running in Colab. Setting up repository...
Cloning into 'Multilingual-Classification-of-Urgent-Disaster-Response-Messages'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 236 (delta 36), reused 210 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (236/236), 18.68 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/Multilingual-Classification-of-Urgent-Disaster-Response-Messages


## **2. Load and Balance Dataset**

In [2]:
import pandas as pd
from sklearn.utils import shuffle
from datasets import Dataset, DatasetDict

In [3]:
from pathlib import Path

if (Path.cwd() / 'datasets').exists():
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path.cwd().parent

In [4]:
DATA_DIR = BASE_DIR / 'datasets'
FILE_PATH = DATA_DIR / 'processed' / 'humaid_processed.csv'

print(f"Project Root Identified as: {BASE_DIR}")
print(f"Looking for file at: {FILE_PATH}")

Project Root Identified as: /content/Multilingual-Classification-of-Urgent-Disaster-Response-Messages
Looking for file at: /content/Multilingual-Classification-of-Urgent-Disaster-Response-Messages/datasets/processed/humaid_processed.csv


In [5]:
if FILE_PATH.exists():
    df = pd.read_csv(FILE_PATH)
    print(f"Successfully loaded {len(df)} rows!")
else:
    print("ERROR: File not found. Check the path printout above.")

Successfully loaded 74227 rows!


In [6]:
df = pd.read_csv(DATA_DIR / 'processed/humaid_processed.csv')

## **3. Data Preparation & 5-Class Mapping**

In [7]:
from datasets import Dataset, DatasetDict

#1. Map the string labels to integers (0 to 4) for the 5 classes
label_mapping = {
    'Critical Rescue': 0,
    'Resource Requests': 1,
    'Situational Awareness': 2,
    'Volunteering and Donations': 3,
    'Irrelevant': 4
}

df['labels'] = df['target_label'].map(label_mapping)

#Drop any rows where the mapping failed or was empty
df = df.dropna(subset=['labels', 'clean_text'])
df['labels'] = df['labels'].astype(int)

In [8]:
#2. Split the data using the existing 'split' column
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'dev']
test_df = df[df['split'] == 'test']


In [9]:
#3. Convert Pandas DataFrames to Hugging Face Datasets
hf_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['clean_text', 'labels']]),
    'validation': Dataset.from_pandas(val_df[['clean_text', 'labels']]),
    'test': Dataset.from_pandas(test_df[['clean_text', 'labels']])
})

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Training samples: 51952
Validation samples: 7564


## **4. Tokenization (XLM-RoBERTa)**

In [10]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    #Truncating to 128 tokens because tweets are short (saves GPU VRAM)
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing datasets...")
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/51952 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Map:   0%|          | 0/14711 [00:00<?, ? examples/s]

## **5. Handling Imbalance with Class Weights**

In [11]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

#Calculate weights based on the 5-class training distribution
train_labels = train_df['labels'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

#Convert weights to a PyTorch tensor and move to GPU if available
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = weights_tensor.to(device)

print(f"Calculated attention weights for 5 classes:\n{class_weights}")

Calculated attention weights for 5 classes:
[1.35591805 5.72473829 0.58922536 0.7121102  1.01379647]


## **6. Model Initialization and Training**

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    # Get the raw predictions from the model
    logits, labels = eval_pred
    # Pick the class with the highest probability
    predictions = np.argmax(logits, axis=-1)

    # Calculate accuracy and F1 (using 'macro' because of your class imbalance!)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')

    return {"accuracy": acc, "f1_score": f1}

In [13]:
from torch import nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [14]:
# 1. Define a Custom Trainer to inject our class weights into the loss function
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Apply CrossEntropyLoss WITH our custom weights
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [15]:
# 2. Load the XLM-RoBERTa Model (Specifying 5 output labels)
print("Loading XLM-RoBERTa model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

Loading XLM-RoBERTa model...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# 3. Set up Training Arguments optimized for Colab's T4 GPU
training_args = TrainingArguments(
    output_dir="./xlm_roberta_disaster",

    # --- NEW LOGGING SETTINGS ---
    eval_strategy="epoch",       # Run the Dev set at the end of every epoch
    logging_strategy="steps",    # Print updates to the console based on steps
    logging_steps=50,            # Print the training loss every 50 steps
    # ----------------------------

    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    report_to="none"             # Keeps logs in Colab, prevents it from asking for wandb login
)

In [17]:
# 4. Initialize our Custom Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics  # <--- ADD THIS LINE HERE
)

In [ ]:
# 5. Start Training
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# 6. Save the final model locally
trainer.save_model("./final_disaster_model_5_classes")
tokenizer.save_pretrained("./final_disaster_model_5_classes")
print("Training complete and model saved!")